# Implementando uma rede simples (MLP) usando o Keras

## Enunciado

Escolha um dataset pronto adequado para classificação binária, evitando datasets "toy" como `Iris` ou `Pima Indians Diabetes`. Certifique-se de selecionar um dataset que ofereça desafios reais em termos de volume e complexidade.


Em seguida, explore o dataset escolhido e explique suas características principais, como o número de amostras, features, e a tarefa de classificação que ele representa.


Desenvolva um modelo sequencial em Keras com uma única camada Dense, utilizando uma unidade com a função de ativação sigmoid. Compile o modelo utilizando o otimizador adam, a função de perda binary_crossentropy, e a métrica accuracy. Inclua também a métrica F1 para uma avaliação mais completa, e explique brevemente a função de cada um desses componentes no treinamento.


Treine o modelo por 50 épocas com um batch size de 10. Após o treinamento, utilize o modelo para prever os rótulos do conjunto de teste e calcule tanto a acurácia quanto a métrica F1. Interprete os resultados, discutindo o desempenho do modelo e possíveis melhorias.

In [15]:
# Importação das bibliotecas essenciais
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K

Nesse primeiro bloco, importei as bibliotecas fundamentais para o desenvolvimento da ponderada:

- **pandas** e **numpy**: Para manipulação e processamento dos dados.
- **sklearn.model_selection**: Para dividir o dataset em conjuntos de treino e teste.
- **sklearn.preprocessing**: Para padronização das características numéricas.
- **sklearn.metrics**: Para cálculo das métricas de avaliação (acurácia e F1 Score).
- **tensorflow.keras**: Para construção e treinamento do modelo de rede neural sequencial.
- **tensorflow.keras.backend**: Para definir a métrica F1 personalizada durante o treinamento.

In [16]:
# Carregar o dataset
path = "Forbes_2000_Companies_2025.csv"
df = pd.read_csv(path, delimiter=';')

# Visualização inicial
print(df.head())
print(df.info())
print(df.describe())


   Rank                                   Company   Headquarters  \
0     1                             JPMorganChase  United States   
1     2                        Berkshire Hathaway  United States   
2     3                                      ICBC          China   
3     4  Saudi Arabian Oil Company (Saudi Aramco)   Saudi Arabia   
4     5                                    Amazon  United States   

               Industry  Sales ($B) Profit ($B) Assets ($B) Market Value ($B)  
0               Banking      285.11      59.36    4,357.86             677.8   
1             Insurance      371.43          89   1,153.88          1,145.46   
2               Banking      221.96      50.84     6,688.6            251.33   
3  Oil & Gas Operations      480.15     104.97      645.03          1,663.38   
4  Retail and Wholesale      637.96      59.25      624.89          2,005.64   
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   

O arquivo `Forbes_2000_Companies_2025.csv` é carregado utilizando o pandas. O delimitador correto (`;`) foi especificado para garantir que os dados sejam lidos corretamente. Em seguida, são realizadas três ações fundamentais de exploração:

- **Visualização das primeiras linhas** (`df.head()`): Permite inspecionar rapidamente a estrutura dos dados e verificar os nomes das colunas.
- **Informações gerais do dataframe** (`df.info()`): Mostra o número de entradas (2,000 empresas), os tipos de dados de cada coluna e se há valores ausentes.
- **Estatísticas descritivas** (`df.describe()`): Ajuda a entender a distribuição dos valores numéricos, como vendas, lucros, ativos e valor de mercado.

A saída revela que o dataset possui:
- Colunas para rank, nome da empresa, país-sede, setor, vendas, lucro, ativos e valor de mercado.
- Quantidades expressivas desses indicadores financeiros, ressaltando o desafio do problema.
- Dados limpos e estruturados, sem entradas nulas nas colunas principais.

Esta etapa é importante para conhecer o conteúdo real da base de dados, identificar possíveis problemas de formatação ou tipos de dados e planejar o pré-processamento. Além disso, fica claro que as variáveis financeiras — como `'Sales ($B)'`, `'Profit ($B)'`, `'Assets ($B)'` e `'Market Value ($B)'` — serão a base para a construção do modelo de classificação binária, como solicitado no enunciado.


In [17]:
# Criação da variável target para classificação binária
# Classe 1: empresas no top 10% do ranking (rank <= 200), classe 0: demais empresas
df['target'] = (df['Rank'] <= 200).astype(int)
print(df['target'].value_counts())

target
0    1800
1     200
Name: count, dtype: int64


Neste passo, os dados são preparados para a tarefa de classificação binária. Foi criada uma nova coluna chamada `target` para representar as classes:

- **Classe 1:** Empresas que estão no top 10% do ranking global, ou seja, aquelas com `Rank` menor ou igual a 200.
- **Classe 0:** As demais empresas fora desse top 10%.

A variável target é um vetor binário, onde 1 indica liderança entre as maiores empresas e 0 indica as demais. 

A contagem das classes mostra um leve desbalanceamento, com:

- 200 empresas na classe positiva (10%)
- 1800 empresas na classe negativa (90%)

Essa definição da variável target transforma o problema original em uma tarefa de classificação binária, exigindo que o modelo aprenda a distinguir as empresas líderes das demais com base em suas métricas financeiras.


In [21]:
# Seleção das features numéricas relevantes para o modelo com nomes corretos
features = ['Sales ($B)', 'Profit ($B)', 'Assets ($B)', 'Market Value ($B)']

for col in features:
    mask = ~df[col].astype(str).str.replace(',', '').str.replace('.', '', 1).str.isnumeric()
    print(f"Valores problemáticos na coluna '{col}':")
    print(df.loc[mask, col])

import re

def fix_decimal(val):
    val = val.replace(',', '')
    # Remove todos os pontos exceto o último:
    if val.count('.') > 1:
        parts = val.split('.')
        return ''.join(parts[:-1]) + '.' + parts[-1]
    return val

for col in features:
    df[col] = df[col].astype(str).apply(fix_decimal)
    df[col] = pd.to_numeric(df[col], errors='coerce')  # Força conversão, vira NaN se não conseguir

X = df[features]
y = df['target']

# Divisão treino e teste (80% treino, 20% teste)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


Valores problemáticos na coluna 'Sales ($B)':
1488   -21.47
Name: Sales ($B), dtype: float64
Valores problemáticos na coluna 'Profit ($B)':
0        59.36 
2        50.84 
3       104.97 
4        59.25 
5        27.85 
         ...   
1977    -0.4843
1980    -0.0285
1986      1.21 
1992       1.2 
1999      1.18 
Name: Profit ($B), Length: 1126, dtype: object
Valores problemáticos na coluna 'Assets ($B)':
0       4,357.86 
1       1,153.88 
2        6,688.6 
3         645.03 
4         624.89 
          ...    
1995        3.16 
1996        1.99 
1997        7.95 
1998       42.56 
1999        3.34 
Name: Assets ($B), Length: 1977, dtype: object
Valores problemáticos na coluna 'Market Value ($B)':
0          677.8 
1       1,145.46 
2         251.33 
3       1,663.38 
4       2,005.64 
          ...    
1991        4.22 
1992        3.52 
1994        1.14 
1995       13.84 
1997        3.93 
Name: Market Value ($B), Length: 1950, dtype: object


Neste bloco, foquei no pré-processamento dos dados financeiros para garantir que todas as features estejam no formato numérico adequado para o modelo:

- Foi verificado se há valores problemáticos nas colunas financeiras, como formatos incorretos que impedem a conversão direta para números.
- Definimos a função `fix_decimal` para corrigir números que contenham múltiplos pontos, mantendo apenas o último como separador decimal e removendo vírgulas que são usadas como separadores de milhar.
- Aplicamos essa função a todas as features selecionadas, convertendo as colunas para numérico com `pd.to_numeric`, transformando valores inválidos em `NaN`.
- Após essa limpeza, os dados ficaram prontos para o treinamento.

Finalmente, o dataset foi dividido em conjuntos de treino e teste, mantendo a proporção original das classes (`stratify=y`), com 80% para treino e 20% para teste. Essa divisão é fundamental para validar o desempenho do modelo em dados nunca vistos durante o treinamento.


In [22]:
# Padronização das features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Neste passo, apliquei a padronização dos dados numéricos usando o `StandardScaler` do scikit-learn. Esse processo é essencial para normalizar as features em uma escala comum com média zero e desvio padrão um.

- O método `fit_transform` é aplicado ao conjunto de treino, ajustando o scaler com base nos dados e transformando-os.
- A mesma transformação é aplicada no conjunto de teste com o método `transform` para garantir que as amostras de teste sejam escalonadas conforme o padrão aprendido no treino.

A padronização é importante para algoritmos baseados em gradiente, como as redes neurais, pois garante que todas as variáveis contribuam de forma equilibrada para o ajuste do modelo, facilitando a convergência no treinamento.


In [23]:
# Definindo o modelo sequencial Keras com uma camada Dense sigmoid
model = Sequential([
    Dense(1, activation='sigmoid', input_shape=(X_train.shape[1],))
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Neste bloco, contrui um modelo de rede neural sequencial simples no Keras com as seguintes características:

- O modelo possui uma única camada densa (Dense) com uma unidade de saída.
- A função de ativação usada é a sigmoid, adequada para problemas de classificação binária, pois retorna uma probabilidade entre 0 e 1.
- O formato da entrada (`input_shape`) é definido pelo número de features no conjunto de treino, garantindo compatibilidade.

Esse modelo básico serve como ponto de partida para a tarefa de classificar as empresas dentro ou fora do top 10% com base nas suas características financeiras.


In [27]:
# Função para cálculo do F1 score em Keras (opcional para monitoramento durante treino)
def f1_metric(y_true, y_pred):
    # Converte ambos para float32 para garantir compatibilidade
    y_true = K.cast(y_true, 'float32')
    y_pred = K.cast(y_pred, 'float32')
    y_pred = K.round(y_pred)
    tp = K.sum(y_true * y_pred)
    predicted_positives = K.sum(y_pred)
    possible_positives = K.sum(y_true)
    precision = tp / (predicted_positives + K.epsilon())
    recall = tp / (possible_positives + K.epsilon())
    return 2 * (precision * recall) / (precision + recall + K.epsilon())

Neste bloco defini uma função para calcular a métrica F1 customizada durante o treinamento do modelo no Keras.

- A função `f1_metric` converte as variáveis verdadeiras (`y_true`) e previstas (`y_pred`) para o tipo float32 para evitar incompatibilidades durante as operações.
- O vetor previsto é arredondado para valores binários (0 ou 1) com `K.round`.
- Calculamos os verdadeiros positivos (`tp`), os positivos previstos (`predicted_positives`) e os positivos reais (`possible_positives`).
- Com esses valores, calculamos a precisão e o recall.
- Por fim, a função retorna o F1 score, que é a média harmônica entre precisão e recall, assegurando o equilíbrio entre os dois.

Essa métrica é importante para avaliar o modelo em cenários de classes desbalanceadas, pois considera tanto falsos positivos quanto falsos negativos, complementando a acurácia tradicional durante o treino.


In [28]:
# Compilação do modelo com Adam e binary_crossentropy
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', f1_metric]
)

Neste bloco, compilei o modelo Keras definindo os seguintes parâmetros:

- **Otimizador:** `adam`, um dos otimizadores mais populares, que adapta dinamicamente a taxa de aprendizado para cada parâmetro durante o treinamento, acelerando a convergência.
- **Função de perda:** `binary_crossentropy`, apropriada para problemas de classificação binária, pois avalia a diferença entre as probabilidades previstas e os rótulos reais.
- **Métricas:** além da acurácia padrão, incluímos a métrica personalizada `f1_metric` para monitorar o desempenho de forma mais completa, especialmente em casos de classes desbalanceadas.

Essa configuração assegura que o treinamento maximize a corretude das previsões enquanto mantém um equilíbrio entre precisão e recall.


In [29]:
# Treinamento do modelo por 50 épocas, batch size 10
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=10,
    verbose=1
)

Epoch 1/50
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 775us/step - accuracy: 0.5744 - f1_metric: 0.1787 - loss: 0.6986 - val_accuracy: 0.9125 - val_f1_metric: 0.2475 - val_loss: 0.6722
Epoch 2/50
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 382us/step - accuracy: 0.9156 - f1_metric: 0.3063 - loss: 0.5973 - val_accuracy: 0.9225 - val_f1_metric: 0.2458 - val_loss: 0.5672
Epoch 3/50
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 367us/step - accuracy: 0.9262 - f1_metric: 0.3250 - loss: 0.5178 - val_accuracy: 0.9325 - val_f1_metric: 0.2725 - val_loss: 0.4854
Epoch 4/50
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 370us/step - accuracy: 0.9375 - f1_metric: 0.4066 - loss: 0.4554 - val_accuracy: 0.9425 - val_f1_metric: 0.3475 - val_loss: 0.4234
Epoch 5/50
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 369us/step - accuracy: 0.9438 - f1_metric: 0.4174 - loss: 0.4052 - val_accuracy: 0.9475 - val_f1_metric: 0.3475 - val_loss: 0.3760
Epoch 6/50
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 370us/step - accuracy: 0.9463 - f1_metric: 0.3966 - loss: 0.3643 - val_accuracy: 0.9500

Neste trecho, o modelo é treinado utilizando o método `fit` do Keras com os seguintes parâmetros:

- **Dados de entrada:** conjuntos de treino `X_train` e os respectivos rótulos `y_train`.
- **Dados de validação:** conjunto de teste `X_test` e rótulos `y_test` para monitorar o desempenho do modelo durante o treinamento.
- **Épocas:** 50, definindo o número de passagens completas pelo conjunto de treino.
- **Batch size:** 10, tamanho dos lotes de amostras processados antes de atualizar os pesos do modelo.
- **Verbose:** 1, para exibir o progresso do treinamento em tempo real.

Esse processo ajusta os pesos do modelo para minimizar a perda, enquanto acompanha métricas relevantes como acurácia e F1 score para avaliação contínua e controle do aprendizado.


In [30]:
# Previsão no conjunto de teste
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

# Cálculo das métricas accuracy e F1 score
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Acurácia no teste: {accuracy:.4f}")
print(f"F1 Score no teste: {f1:.4f}")

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
Acurácia no teste: 0.9525
F1 Score no teste: 0.7077


Neste último bloco, o modelo treinado é usado para prever as classes no conjunto de teste:

- As probabilidades previstas (`y_pred_prob`) são convertidas em classes binárias utilizando o limiar de 0.5.
- Calculamos as métricas de avaliação principais: acurácia e F1 score.

### Resultados Obtidos

- **Acurácia no teste: 95,25%**
- **F1 Score no teste: 0,7077**

### Interpretação

- O modelo apresenta alta acurácia, indicando que a maioria das previsões está correta.
- Entretanto, o F1 score é significativamente menor, sugerindo que há um desequilíbrio entre precisão e recall — o modelo pode estar falhando em captar bem uma das classes (provavelmente a minoritária).
  
### Possíveis Melhorias

- Aplicar técnicas de balanceamento de classes, como oversampling das minoritárias ou undersampling das majoritárias, para melhorar o equilíbrio do modelo.
- Experimentar modelos mais complexos, adicionando camadas ou unidades na rede neural para capturar padrões não lineares mais sofisticados.
- Afinar o threshold da sigmoide para melhorar o equilíbrio entre falsos positivos e falsos negativos.
- Investir em engenharia de features para extrair informações adicionais que auxiliem na discriminação das classes.

Essas ações podem ajudar a melhorar a capacidade de classificação do modelo, principalmente considerando o desafio real apresentado pelo dataset.
